<a href="https://colab.research.google.com/github/SahitiVuppala/Pfizer-AI-Document-Intelligence-/blob/main/Pfizer_AI_Intelligence_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio
!pip install -q pymupdf PyPDF2
!pip install -q sentence-transformers transformers sentencepiece
!pip install -q faiss-cpu
!pip install -q numpy pandas
!pip install -q pyngrok

print("All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.3 MB/s eta 0:00:00
All packages installed.


In [ ]:
import gradio as gr
import fitz  # PyMuPDF
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import json
from datetime import datetime
import re

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


print("Loading generation model (flan-t5-base)...")
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model  = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
gen_model.eval()

def llm(prompt: str, max_new_tokens: int = 300) -> str:
    """Run flan-t5-base directly — no pipeline() needed."""
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Imports and configuration complete.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Imports and configuration complete.


In [ ]:
@dataclass
class PageInfo:
    """Stores information about a single page."""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF."""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = field(default_factory=list)

@dataclass
class ChunkMetadata:
    """Rich metadata for each text chunk."""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

In [ ]:
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

# Keyword signals
DOC_TYPE_KEYWORDS = [
    ("Cover Letter",           ["to whom it may concern", "dear", "sincerely", "storage condition", "re:"]),
    ("Certificate Of Quality", ["certificate of quality", "lot number", "expiration date", "date of manufacture",
                                 "product release", "batch", "autoclave", "gamma irradiation", "conforms"]),
    ("Packaging Specification",["packaging component", "packaging specification", "blister", "tyvek",
                                 "ecn number", "change history", "pkg-spec", "corrugated"]),
    ("Bse/Tse Declaration",    ["bse", "tse", "spongiform", "animal origin", "bovine", "porcine",
                                 "transmissible", "encephalopathies"]),
    ("Material Description",   ["material description", "materials of construction", "sterilization compatibility",
                                 "physical properties", "polypropylene", "epdm", "peek", "borosilicate"]),
    ("Supplier Qualification", ["supplier qualification", "qualification record", "iso 9001", "iso 13485",
                                 "audit", "certification", "approved supplier", "fda establishment"]),
    ("Chain Of Custody",       ["chain of custody", "assemblies manufactured", "traceability",
                                 "shipment flow", "distribution center", "eysins", "lot number traceable"]),
]

def classify_document_type(text: str) -> str:
    """Classify page into a pharmaceutical document type using keyword heuristics."""
    text_lower = text.lower()
    best_type = "Other"
    best_score = 0
    for doc_type, keywords in DOC_TYPE_KEYWORDS:
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > best_score:
            best_score = score
            best_type = doc_type
    return best_type

def detect_document_boundary(prev_text: str, curr_text: str,
                             current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if they are from the same document.
    Uses title-change and continuation signals.
    """
    if not prev_text or not curr_text:
        return False


    curr_lower = curr_text.lower()[:300]
    continuation_signals = ["continued", "page 2 of", "page 3 of", "(continued)", "cont'd"]
    if any(s in curr_lower for s in continuation_signals):
        return True


    curr_type = classify_document_type(curr_text)
    if curr_type != current_doc_type and curr_type != "Other":
        return False  # Different doc type = new document


    return curr_type == "Other" or curr_type == current_doc_type

In [ ]:
def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    """
    print("Starting PDF extraction and analysis...")

    # Open PDF with PyMuPDF
    if isinstance(pdf_file, str):
        doc = fitz.open(pdf_file)
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()
        # OCR fallback for scanned pages (requires pytesseract)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                from PIL import Image
                import pytesseract
                import io
                pix = page.get_pixmap()
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
                print(f"  Page {i}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""
        pages_info.append(PageInfo(page_num=i, text=text))
    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Classify pages
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            prev_text = pages_info[i - 1].text
            is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)
            if is_same:
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # Save document
                logical_docs.append(LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text="\n\n".join(p.text for p in current_doc_pages)
                ))
                doc_counter += 1
                current_doc_type = classify_document_type(page_info.text)
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # Append
    if current_doc_pages:
        logical_docs.append(LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join(p.text for p in current_doc_pages)
        ))

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

In [ ]:
def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                  chunk_size: int = 500,
                                  overlap: int = 100) -> List[ChunkMetadata]:
    """Chunk a logical document while preserving rich metadata."""
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        chunks_metadata.append(ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        ))
    else:
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = " ".join(words[start_idx:end_idx])
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)
            chunks_metadata.append(ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            ))
            if end_idx >= len(words):
                break

    return chunks_metadata


def process_all_documents(logical_docs: List[LogicalDocument]) -> List[ChunkMetadata]:
    """Process all logical documents into chunks with metadata."""
    all_chunks = []
    for logical_doc in logical_docs:
        chunks = chunk_document_with_metadata(logical_doc)
        logical_doc.chunks = chunks
        all_chunks.extend(chunks)
        print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks")
    return all_chunks

In [ ]:
QUERY_ROUTE_KEYWORDS = [
    ("Certificate Of Quality", ["lot number", "batch", "expir", "manufactur", "test result",
                                  "autoclave", "gamma", "steriliz", "conforms", "release"]),
    ("Packaging Specification", ["packaging", "part number", "blister", "carton", "tyvek",
                                  "material", "ecn", "drawing", "configuration"]),
    ("Bse/Tse Declaration",     ["bse", "tse", "bse/tse", "animal origin", "bovine", "porcine",
                                  "spongiform", "transmissible", "compliance date",
                                  "valid until", "risk assessment", "encephalopathies"]),
    ("Material Description",    ["material description", "construction", "polypropylene",
                                  "epdm", "peek", "borosilicate", "silicone", "physical"]),
    ("Supplier Qualification",  ["supplier", "audit", "iso 9001", "iso 13485", "certif",
                                  "qualification", "approved", "vendor", "sub-supplier"]),
    ("Chain Of Custody",        ["chain of custody", "traceab", "assembly", "shipment",
                                  "distribution", "manufactured at", "custody"]),
    ("Cover Letter",            ["storage", "temperature", "operating", "letter", "recommend"]),
]

def predict_query_document_type(query: str) -> Tuple[str, float]:
    """Predict which pharmaceutical document type best matches the query."""
    q = query.lower()
    best_type = "Other"
    best_score = 0
    for doc_type, keywords in QUERY_ROUTE_KEYWORDS:
        score = sum(1 for kw in keywords if kw in q)
        if score > best_score:
            best_score = score
            best_type = doc_type

    confidence = min(0.5 + best_score * 0.2, 0.95) if best_score > 0 else 0.0
    return best_type, confidence


class IntelligentRetriever:
    """Advanced retrieval system with FAISS and per-doc-type index routing."""

    def __init__(self):
        self.index = None
        self.chunks_metadata: List[ChunkMetadata] = []
        self.doc_type_indices = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """Build FAISS indices with document type segregation."""
        print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = embed_model.encode(texts, show_progress_bar=True,
                                        convert_to_numpy=True).astype("float32")

        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)


        doc_types = set(c.doc_type for c in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, c in enumerate(chunks_metadata) if c.doc_type == doc_type]
            if type_indices:
                type_embs = embeddings[type_indices]
                type_idx = faiss.IndexFlatL2(dim)
                type_idx.add(type_embs)
                self.doc_type_indices[doc_type] = {"index": type_idx, "mapping": type_indices}

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def retrieve(self, query: str, k: int = 4,
                 filter_doc_type: Optional[str] = None,
                 auto_route: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """Retrieve relevant chunks with optional filtering and auto-routing."""
        query_emb = embed_model.encode([query], convert_to_numpy=True).astype("float32")

        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            type_data = self.doc_type_indices[filter_doc_type]
            D, I = type_data["index"].search(query_emb, k)
            chunk_indices = [type_data["mapping"][i] for i in I[0]]
            distances = D[0]
        elif auto_route:
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")
            if confidence > 0.65 and predicted_type in self.doc_type_indices:
                type_data = self.doc_type_indices[predicted_type]
                D, I = type_data["index"].search(query_emb, k)
                chunk_indices = [type_data["mapping"][i] for i in I[0]]
                distances = D[0]
            else:
                D, I = self.index.search(query_emb, k)
                chunk_indices = I[0]
                distances = D[0]
        else:
            D, I = self.index.search(query_emb, k)
            chunk_indices = I[0]
            distances = D[0]

        max_dist = max(distances) if distances.max() > 0 else 1.0
        scores = [(max_dist - d) / max_dist for d in distances]
        return [(self.chunks_metadata[i], scores[idx]) for idx, i in enumerate(chunk_indices)]

In [ ]:
import re as _re

def extract_direct_answer(query: str, context: str) -> str:
    """
    Fast keyword-based extraction for common pharma query patterns.
    Runs before flan-t5 as a high-precision fallback for structured fields.
    """
    q = query.lower()
    lines = context.split("\n")

    # Date/compliance queries
    if any(w in q for w in ["date", "valid", "expire", "compliance", "assessment"]):
        date_lines = [l.strip() for l in lines
                      if _re.search(r"\d{1,2}\s+\w+\s+\d{4}|\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}", l)
                      and len(l.strip()) > 5]
        if date_lines:
            return "\n".join(date_lines[:4])

    # Lot / batch number queries
    if any(w in q for w in ["lot", "batch", "lot number", "batch number"]):
        lot_lines = [l.strip() for l in lines
                     if _re.search(r"lot\s*(number)?[:\s]+|batch\s*(number)?[:\s]+|lot no", l, _re.I)
                     and len(l.strip()) > 5]
        if lot_lines:
            return "\n".join(lot_lines[:4])

    # Part number queries
    if any(w in q for w in ["part number", "article number", "part no"]):
        part_lines = [l.strip() for l in lines
                      if _re.search(r"pkg-|part number|article number|pkg\s*no", l, _re.I)
                      and len(l.strip()) > 5]
        if part_lines:
            return "\n".join(part_lines[:4])

    return ""  # No direct extraction match


def generate_answer_with_sources(query: str,
                                  retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """Generate answer with source attribution using flan-t5 + direct extraction fallback."""
    if not retrieved_chunks:
        return {"answer": "I could not find relevant information to answer your question.",
                "sources": [], "confidence": 0.0}

    sources = []
    context_parts = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(chunk_meta.text[:600])
        sources.append({
            "doc_type": chunk_meta.doc_type,
            "pages": f"{chunk_meta.page_start + 1}-{chunk_meta.page_end + 1}",
            "relevance": f"{score:.0%}",
            "preview": chunk_meta.text[:100] + "..."
        })

    context = " ".join(context_parts)[:900]
    page_nums = ", ".join(str(c.page_start + 1) for c, _ in retrieved_chunks)
    avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

    # Try direct keyword extraction first (fast, precise for structured fields)
    direct = extract_direct_answer(query, " \n ".join(c.text for c, _ in retrieved_chunks))
    if direct:
        return {
            "answer": direct + f"\n\n_(Source: page(s) {page_nums})_",
            "sources": sources,
            "confidence": avg_score,
            "chunks_used": len(retrieved_chunks)
        }

    # Fall back to flan-t5 for open-ended questions
    prompt = (
        "Use the text below to answer the question. "
        "Copy exact values like numbers, dates, and codes as they appear. "
        "If the answer is not in the text, say: not found.\n\n"
        "Text: " + context + "\n\n"
        "Question: " + query
    )

    try:
        answer = llm(prompt, max_new_tokens=200).strip()
        # If flan-t5 still says it doesn't know, scan context for key sentences
        if any(phrase in answer.lower() for phrase in ["not found", "i don't know", "i do not know", "cannot find"]):
            # Return the most relevant sentence from the top chunk instead
            top_text = retrieved_chunks[0][0].text
            sentences = [s.strip() for s in top_text.replace("\n", " ").split(".") if len(s.strip()) > 20]
            q_words = set(query.lower().split())
            scored = [(sum(1 for w in q_words if w in s.lower()), s) for s in sentences]
            scored.sort(reverse=True)
            best = scored[0][1] if scored else top_text[:300]
            answer = f"Based on the document: {best}."

        return {
            "answer": answer + f"\n\n_(Source: page(s) {page_nums})_",
            "sources": sources,
            "confidence": avg_score,
            "chunks_used": len(retrieved_chunks)
        }
    except Exception as e:
        return {"answer": f"Error generating answer: {e}", "sources": sources, "confidence": 0.0}

In [ ]:
class EnhancedDocumentStore:
    """Manages the complete document processing and retrieval pipeline."""

    def __init__(self):
        self.pages_info: List[PageInfo] = []
        self.logical_docs: List[LogicalDocument] = []
        self.chunks_metadata: List[ChunkMetadata] = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """Complete PDF processing pipeline."""
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()
        try:
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)
            self.chunks_metadata = process_all_documents(self.logical_docs)
            self.retriever.build_indices(self.chunks_metadata)
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                "filename": filename,
                "total_pages": len(self.pages_info),
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunks_metadata),
                "document_types": list(set(d.doc_type for d in self.logical_docs)),
                "processing_time": f"{process_time:.1f}s"
            }
            self.is_ready = True
            return True, self.processing_stats
        except Exception as e:
            return False, {"error": str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
              auto_route: bool = True, k: int = 4) -> Dict:
        """Query the document store."""
        if not self.is_ready:
            return {"answer": "Please upload and process a PDF first.",
                    "sources": [], "confidence": 0.0}
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route and filter_type is None
        )
        result = generate_answer_with_sources(question, retrieved)
        result["filter_used"] = filter_type or ("auto" if auto_route else "none")
        return result

    def get_document_structure(self) -> List[Dict]:
        """Get the document structure for UI display."""
        return [
            {
                "id": doc.doc_id,
                "type": doc.doc_type,
                "pages": f"{doc.page_start + 1}-{doc.page_end + 1}",
                "chunks": len(doc.chunks) if doc.chunks else 0,
                "preview": doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            }
            for doc in self.logical_docs
        ]

# Global store
doc_store = EnhancedDocumentStore()

In [ ]:
def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file", "", gr.update(choices=["All"])
    filename = pdf_file.split("/")[-1] if isinstance(pdf_file, str) else getattr(pdf_file, "name", "document.pdf")
    success, stats = doc_store.process_pdf(pdf_file, filename=filename)
    if success:
        status_msg = (
            f"**Successfully Processed:**\n"
            f"- File: {stats['filename']}\n"
            f"- Pages: {stats['total_pages']}\n"
            f"- Documents Found: {stats['documents_found']}\n"
            f"- Chunks Created: {stats['total_chunks']}\n"
            f"- Types: {', '.join(stats['document_types'])}\n"
            f"- Time: {stats['processing_time']}"
        )
        structure = doc_store.get_document_structure()
        structure_display = "\n".join(
            f"- **{d['type']}** (Pages {d['pages']}): {d['chunks']} chunks"
            for d in structure
        )
        doc_types = ["All"] + stats["document_types"]
        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", "", gr.update(choices=["All"])


def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions using Gradio 6.0 message dict format."""
    if not message or not message.strip():
        return history
    if not doc_store.is_ready:
        return history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "Please upload and process a pharmaceutical PDF first."}
        ]
    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(message, filter_type=filter_type,
                              auto_route=auto_route and filter_type is None,
                              k=int(num_chunks))
    response = result["answer"] + "\n\n"
    if result["sources"]:
        response += "**Sources:**\n"
        for src in result["sources"]:
            response += f"- {src['doc_type']} (Pages {src['pages']}) \u2014 Relevance: {src['relevance']}\n"
    response += f"\n*Confidence: {result['confidence']:.0%} | Chunks used: {result.get('chunks_used', 0)} | Filter: {result.get('filter_used', 'auto')}*"
    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]


def export_chat(history):
    """Export chat history as a downloadable .txt file."""
    if not history:
        return gr.update(visible=False)
    lines = ["Pharmaceutical Document Q&A — Chat Export",
             f"Exported: {datetime.now().isoformat()}", "=" * 50, ""]
    for msg in history:
        prefix = "Q" if msg["role"] == "user" else "A"
        lines.append(f"{prefix}: {msg['content']}\n")
    path = "/content/chat_export.txt"
    open(path, "w").write("\n".join(lines))
    return gr.update(value=path, visible=True)


def update_status_bar():
    if doc_store.is_ready:
        s = doc_store.processing_stats
        return (f"**Status:** Ready | **Documents:** {s.get('documents_found',0)} | "
                f"**Chunks:** {s.get('total_chunks',0)}")
    return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"


def create_interface():
    with gr.Blocks(title="Pharmaceutical Document Q&A System") as demo:
        gr.Markdown("""
        # \U0001f9ea Pharmaceutical Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        Upload a pharmaceutical blob PDF to identify document types, build a searchable
        index, and ask questions in natural language. Powered by open-source models only.
        """)

        with gr.Row():
            # Left
            with gr.Column(scale=2):
                pdf_input = gr.File(label="\U0001f4ce Upload Pharmaceutical PDF",
                                    file_types=[".pdf"], type="filepath")
                with gr.Row():
                    process_btn = gr.Button("Process Document", variant="primary", scale=2)
                    clear_all_btn = gr.Button("Clear All", variant="secondary", scale=1)

            # Middle
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(value="Waiting for PDF upload...")
                structure_output = gr.Markdown(value="")
                gr.Markdown("### Retrieval Settings")
                doc_filter = gr.Dropdown(choices=["All"], value="All",
                                         label="Document Type Filter",
                                         info="Filter to a specific pharmaceutical document type")
                auto_route = gr.Checkbox(value=True, label="Auto-Route Queries",
                                         info="Automatically detect the most relevant document type")
                num_chunks = gr.Slider(minimum=1, maximum=10, value=4, step=1,
                                       label="Chunks to Retrieve")

            # Right
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(label="Conversation", height=450,
                                     show_label=False)
                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="e.g. What is the lot number? What sterilization method was used?",
                        show_label=False, scale=4)
                    send_btn = gr.Button("Send", scale=1, variant="primary")
                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm")
                    example_btn1 = gr.Button("Summarise this document", size="sm")
                    example_btn2 = gr.Button("Find lot numbers", size="sm")
                with gr.Row():
                    export_btn = gr.Button("\u2b07\ufe0f Export Chat", size="sm")
                    export_file = gr.File(visible=False)

        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0")



        def process_with_status(pdf):
            s, st, f = process_pdf_handler(pdf)
            return s, st, f, update_status_bar()

        def chat_with_status(msg, hist, filt, ar, nc):
            return chat_handler(msg, hist, filt, ar, nc), update_status_bar()

        def clear_all():
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (None, "Waiting for PDF upload...", "",
                    gr.update(choices=["All"], value="All"), [], "", update_status_bar())

        process_btn.click(fn=process_with_status, inputs=[pdf_input],
                          outputs=[status_output, structure_output, doc_filter, status_bar])
        pdf_input.change(fn=process_with_status, inputs=[pdf_input],
                         outputs=[status_output, structure_output, doc_filter, status_bar])
        clear_all_btn.click(fn=clear_all,
                            outputs=[pdf_input, status_output, structure_output,
                                     doc_filter, chatbot, msg_input, status_bar])

        send_btn.click(fn=chat_with_status,
                       inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
                       outputs=[chatbot, status_bar]).then(lambda: "", outputs=[msg_input])
        msg_input.submit(fn=chat_with_status,
                         inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
                         outputs=[chatbot, status_bar]).then(lambda: "", outputs=[msg_input])

        clear_chat_btn.click(lambda: [], outputs=[chatbot])

        example_btn1.click(
            fn=lambda h: chat_handler("Can you summarise the main points in this document?",
                                      h, "All", True, 4),
            inputs=[chatbot], outputs=[chatbot])
        example_btn2.click(
            fn=lambda h: chat_handler("What lot numbers or batch numbers are mentioned?",
                                      h, "All", True, 4),
            inputs=[chatbot], outputs=[chatbot])

        export_btn.click(fn=export_chat, inputs=[chatbot], outputs=[export_file])

    return demo

In [ ]:
import threading, time, socket
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
ngrok.kill()


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]

port = find_free_port()
print(f"Using port {port}")

# Open ngrok
public_url = ngrok.connect(port, bind_tls=True)
print(f"\n\u2705 Open your chatbot here: {public_url}\n")

demo = create_interface()

# Launch Gradio
def run_gradio():
    demo.launch(
        server_name="0.0.0.0",
        server_port=port,
        share=False,
        debug=False,
        quiet=True
    )

t = threading.Thread(target=run_gradio, daemon=True)
t.start()
time.sleep(3)
print("Gradio is running. Open the link above.")

Using port 44535

✅ Open your chatbot here: NgrokTunnel: "https://thrash-trio-clash.ngrok-free.dev" -> "http://localhost:44535"



<IPython.core.display.Javascript object>

Gradio is running. Open the link above.
